### **Corrective RAG(CRAG)**

In [ ]:
from typing import List, TypedDict
import time
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from dotenv import load_dotenv

load_dotenv()
docs = (
    PyPDFLoader("./documents/book1.pdf").load() +
    PyPDFLoader("./documents/book2.pdf").load() +
    PyPDFLoader("./documents/book3.pdf").load()
)
len(docs)

In [ ]:
chunks = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150).split_documents(docs)

for d in chunks:
  d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(
  search_type="similarity",
  search_kwargs={"k": 4}
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
class State(TypedDict):
    question: str
    docs: List[Document]
    answer: str

def retrieve(state):
    q = state["question"]
    return {"docs": retriever.invoke(q)}

prompt = ChatPromptTemplate.from_messages(
  [
    ("system", "Answer only from the context. If not in context, say you don't know."),
    ("human", "Question: {question}\n\nContext:\n{context}"),
  ]
)
def generate(state):
  context = "\n\n".join(d.page_content for d in state["docs"])
  out = (prompt | llm).invoke({"question": state["question"], "context": context})
  return {"answer": out.content}

g = StateGraph(State)
g.add_node("retrieve", retrieve)
g.add_node("generate", generate)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", END)
app = g.compile()

app

In [ ]:
res = app.invoke({"question": "WHat is a transformer in deep learning.", "docs": [], "answer": ""})
print(res["answer"])
print(res['docs'][0].page_content)
print('*'*100)
print(res['docs'][1].page_content)
print('*'*100)
print(res['docs'][2].page_content)
print('*'*100)
print(res['docs'][3].page_content)

In [ ]:
docs = (
    PyPDFLoader("./documents/book1.pdf").load()
    + PyPDFLoader("./documents/book2.pdf").load()
    + PyPDFLoader("./documents/book3.pdf").load()
)

chunks = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150).split_documents(docs)
for d in chunks:
  d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(
  search_type="similarity",
  search_kwargs={"k": 4}
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class State(TypedDict):
    question: str
    docs: List[Document]
    strips: List[str]
    kept_strips: List[str]
    refined_context: str
    answer: str

def retrieve(state: State) -> State:
    q = state["question"]
    return {"docs": retriever.invoke(q)}

def decompose_to_sentences(text: str) -> List[str]:
  text = re.sub(r"\s+", " ", text).strip()
  sentences = re.split(r"(?<=[.!?])\s+", text)
  return [s.strip() for s in sentences if len(s.strip()) > 20]

class KeepOrDrop:
  keep:bool

filter_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are a strict relevance filter.\n"
      "Return keep=true only if the sentence directly helps answer the question.\n"
      "Use ONLY the sentence. Output JSON only.",
    ),
    ("human", "Question: {question}\n\nSentence:\n{sentence}"),
  ]
)

filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop)

def refine(state: State) -> State:
  q = state["question"]
  context = "\n\n".join(d.page_content for d in state["docs"]).strip()
  strips = decompose_to_sentences(context)
  kept: List[str] = []

  for s in strips:
    if filter_chain.invoke({"question": q, "sentence": s}).keep:
      kept.append(s)

  refined_context = "\n".join(kept).strip()
  return {
    "strips": strips,
    "kept_strips": kept,
    "refined_context": refined_context,
  }

answer_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are a helpful ML tutor. Answer ONLY using the provided refined bullets.\n"
      "If the bullets are empty or insufficient, say: 'I don't know based on the provided books.'",
    ),
    ("human", "Question: {question}\n\nRefined context:\n{refined_context}"),
  ]
)

def generate(state: State) -> State:
  out = (answer_prompt | llm).invoke({"question": state["question"], "refined_context": state['refined_context']})

  return {"answer": out.content}

g = StateGraph(State)
g.add_node("retrieve", retrieve)
g.add_node("refine", refine)
g.add_node("generate", generate)

g.add_edge(START, "retrieve")
g.add_edge("retrieve", "refine")
g.add_edge("refine", "generate")
g.add_edge("generate", END)
app = g.compile()

app

In [ ]:
res = app.invoke({
    "question": "Explain the bias–variance tradeoff",
    "docs": [],
    "strips": [],
    "kept_strips": [],
    "refined_context": "",
    "answer": ""
})
print(res["answer"])
print(res['docs'][0].page_content)
print('*'*100)
print(res['docs'][1].page_content)
print('*'*100)
print(res['docs'][2].page_content)
print('*'*100)
print(res['docs'][3].page_content)
print(res['refined_context'])

In [ ]:
def decompose_to_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]

decompose_to_sentences("""A transformer in deep learning is a type of model architecture that is particularly effective for processing sequential data, such as text. It utilizes mechanisms called self-attention and feedforward neural networks to weigh the importance of different parts of the input data, allowing it to capture long-range dependencies and relationships within the data. Unlike traditional recurrent neural networks (RNNs), transformers do not process data sequentially, which enables them to be more parallelizable and efficient in training. This architecture has become foundational in natural language processing tasks and has led to significant advancements in the field.""")

### **Retrieval Evaluator**

In [ ]:
docs = (
    PyPDFLoader("./documents/book1.pdf").load()
    + PyPDFLoader("./documents/book2.pdf").load()
    + PyPDFLoader("./documents/book3.pdf").load()
)

chunks = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150).split_documents(docs)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
UPPER_TH = 0.7
LOWER_TH = 0.3

class State(TypedDict):
    question: str
    docs: List[Document]
    good_docs: List[Document]
    verdict: str
    reason: str
    strips: List[str]
    kept_strips: List[str]
    web_docs: List[Document]
    web_query: str
    refined_context: str
    answer: str

def retrieve_node(state: State) -> State:
    q = state["question"]
    return {"docs": retriever.invoke(q)}

class DocEvalScore(BaseModel):
    score: float
    reason: str

doc_eval_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are a strict retrieval evaluator for RAG.\n"
      "You will be given ONE retrieved chunk and a question.\n"
      "Return a relevance score in [0.0, 1.0].\n"
      "- 1.0: chunk alone is sufficient to answer fully/mostly\n"
      "- 0.0: chunk is irrelevant\n"
      "Be conservative with high scores.\n"
      "Also return a short reason.\n"
      "Output JSON only.",
    ),
    ("human", "Question: {question}\n\nChunk:\n{chunk}"),
  ]
)

doc_eval_chain = doc_eval_prompt | llm.with_structured_output(DocEvalScore)

def eval_each_doc_node(state: State) -> State:
  q = state["question"]
  scores: List[float] = []
  reasons: List[str] = []
  good: List[Document] = []

  for d in state["docs"]:
    out = doc_eval_chain.invoke({"question": q, "chunk": d.page_content})
    scores.append(out.score)
    reasons.append(out.reason)
    if out.score > LOWER_TH:
      good.append(d)

  if any(s > UPPER_TH for s in scores):
    return {
        "good_docs": good,
        "verdict": "CORRECT",
        "reason": f"At least one retrieved chunk scored > {UPPER_TH}.",
    }

  if len(scores) > 0 and all(s < LOWER_TH for s in scores):
    why = "No chunk was sufficient."
    return {
      "good_docs": [],
      "verdict": "INCORRECT",
      "reason": f"All retrieved chunks scored < {LOWER_TH}. {why}",
    }

  why = "Mixed relevance signals."
  return {
    "good_docs": good,
    "verdict": "AMBIGUOUS",
    "reason": f"No chunk scored > {UPPER_TH}, but not all were < {LOWER_TH}. {why}",
  }

def decompose_to_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]

class KeepOrDrop(BaseModel):
    keep: bool

filter_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are a strict relevance filter.\n"
      "Return keep=true only if the sentence directly helps answer the question.\n"
      "Use ONLY the sentence. Output JSON only.",
    ),
    ("human", "Question: {question}\n\nSentence:\n{sentence}"),
  ]
)

def refine(state: State) -> State:
  q = state["question"]
  context = "\n\n".join(d.page_content for d in state["good_docs"]).strip()
  strips = decompose_to_sentences(context)
  kept: List[str] = []

  for s in strips:
    if filter_prompt.invoke({"question": q, "sentence": s}).keep:
      kept.append(s)

  refined_context = "\n".join(kept).strip()
  return {
    "strips": strips,
    "kept_strips": kept,
    "refined_context": refined_context,
  }

answer_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are a helpful ML tutor. Answer ONLY using the provided context.\n"
      "If the context is empty or insufficient, say: 'I don't know.'",
    ),
    ("human", "Question: {question}\n\nRefined context:\n{refined_context}"),
  ]
)

def generate(state: State) -> State:
  out = (answer_prompt | llm).invoke(
    {"question": state["question"], "refined_context": state["refined_context"]}
  )

  return {"answer": out.content}

def fail_node(state: State) -> State:
  return {"answer": f"FAIL: {state['reason']}"}

def ambiguous_node(state: State) -> State:
  return {"answer": f"Ambiguous: {state['reason']}"}

def route_after_eval(state: State) -> str:
  if state["verdict"] == "CORRECT":
    return "refine"
  elif state["verdict"] == "INCORRECT":
    return "web_search"
  else:
     return "ambiguous"

g = StateGraph(State)
g.add_node("retrieve", retrieve_node)
g.add_node("eval_each_doc", eval_each_doc_node)
g.add_node("refine", refine)
g.add_node("generate", generate)
g.add_node("fail", fail_node)
g.add_node("ambiguous", ambiguous_node)

g.add_edge(START, "retrieve")
g.add_edge("retrieve", "eval_each_doc")
g.add_conditional_edges(
    "eval_each_doc",
    route_after_eval,
    {
        "refine": "refine",
        "web_search": "fail",
        "ambiguous": "ambiguous",
    },
)
g.add_edge("refine", "generate")
g.add_edge("generate", END)
g.add_edge("fail", END)
g.add_ede("ambigous", END)

app = g.compile()
app

In [ ]:
res = app.invoke(
  {
    "question": "What are attention mechanisms and why are they important in current models?",
    "docs": [],
    "good_docs": [],
    "verdict": "",
    "reason": "",
    "strips": [],
    "kept_strips": [],
    "refined_context": "",
    "answer": "",
  }
)

print("VERDICT:", res["verdict"])
print("REASON:", res["reason"])
print("\nOUTPUT:\n", res["answer"])

In [ ]:
class DocEvalScore(BaseModel):
  score: float
  reason: str

doc_eval_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are a strict retrieval evaluator for RAG.\n"
      "You will be given ONE retrieved chunk and a question.\n"
      "Return a relevance score in [0.0, 1.0].\n"
      "- 1.0: chunk alone is sufficient to answer fully/mostly\n"
      "- 0.0: chunk is irrelevant\n"
      "Be conservative with high scores.\n"
      "Also return a short reason.\n"
      "Output JSON only.",
    ),
    ("human", "Question: {question}\n\nChunk:\n{chunk}"),
  ]
)

def eval_each_doc_node(state: State) -> State:
  q = state["question"]
  scores: List[float] = []
  good: List[Document] = []

  for d in state["docs"]:
    out = doc_eval_chain.invoke({"question": q, "chunk": d.page_content})
    scores.append(out.score)

    if out.score > LOWER_TH:
      good.append(d)

  if any(s > UPPER_TH for s in scores):
    return {
      "good_docs": good,
      "verdict": "CORRECT",
      "reason": f"At least one retrieved chunk scored > {UPPER_TH}.",
    }

  if len(scores) > 0 and all(s < LOWER_TH for s in scores):
    why = "No chunk was sufficient."
    return {
      "good_docs": [],
      "verdict": "INCORRECT",
      "reason": f"All retrieved chunks scored < {LOWER_TH}. {why}",
    }

  why = "Mixed relevance signals."
  return {
    "good_docs": good,
    "verdict": "AMBIGUOUS",
    "reason": f"No chunk scored > {UPPER_TH}, but not all were < {LOWER_TH}. {why}",
  }

def decompose_to_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]

class KeepOrDrop(BaseModel):
    keep: bool

filter_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are a strict relevance filter.\n"
      "Return keep=true only if the sentence directly helps answer the question.\n"
      "Use ONLY the sentence. Output JSON only.",
    ),
    ("human", "Question: {question}\n\nSentence:\n{sentence}"),
  ]
)

filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop)
def refine(state: State) -> State:
  q = state["question"]
  if state.get("verdict") == "CORRECT":
    context = "\n\n".join(d.page_content for d in state["good_docs"]).strip()
  else:
    context = "\n\n".join(d.page_content for d in state["web_docs"]).strip()

  strips = decompose_to_sentences(context)
  kept: List[str] = []

  for s in strips:
    if filter_chain.invoke({"question": q, "sentence": s}).keep:
      kept.append(s)

  refined_context = "\n".join(kept).strip()
  return {
    "strips": strips,
    "kept_strips": kept,
    "refined_context": refined_context,
  }

class WebQuery(BaseModel):
  query: str

rewrite_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "Rewrite the user question into a web search query composed of keywords.\n"
      "Rules:\n"
      "- Keep it short (6–14 words).\n"
      "- If the question implies recency (e.g., recent/latest/last week/last month), add a constraint like (last 30 days).\n"
      "- Do NOT answer the question.\n"
      "- Return JSON with a single key: query",
    ),
    ("human", "Question: {question}"),
  ]
)

rewrite_chain = rewrite_prompt | llm.with_structured_output(WebQuery)
def rewrite_query_node(state: State) -> State:
  out = rewrite_chain.invoke({"question": state["question"]})
  return {"web_query": out.query}

from tavily import TavilySearchResults

tavily = TavilySearchResults(max_results=5)
def web_search_node(state: State) -> State:
  q = state.get("web_query") or state["question"]
  results = tavily.invoke({"query": q})

  web_docs = []
  for r in results or []:
    title = r.get("title", "")
    url = r.get("url", "")
    content = r.get("content", "") or r.get("snippet", "")
    text = f"TITLE: {title}\nURL: {url}\nCONTENT:\n{content}"
    web_docs.append(Document(page_content=text, metadata={"url": url, "title": title}))

  return {"web_docs": web_docs}

answer_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are a helpful ML tutor. Answer ONLY using the provided context.\n"
      "If the context is empty or insufficient, say: 'I don't know.'",
    ),
    ("human", "Question: {question}\n\nRefined context:\n{refined_context}"),
  ]
)

def generate(state: State) -> State:
  out = (answer_prompt | llm).invoke(
    {"question": state["question"], "refined_context": state["refined_context"]}
  )

  return {"answer": out.content}

def fail_node(state: State) -> State:
  return {"answer": f"FAIL: {state['reason']}"}

def ambiguous_node(state: State) -> State:
  return {"answer": f"Ambiguous: {state['reason']}"}

def route_after_eval(state: State) -> str:
  if state["verdict"] == "CORRECT":
    return "refine"
  elif state["verdict"] == "INCORRECT":
    return "rewrite_query"
  else:
     return "ambiguous"

g = StateGraph(State)
g.add_node("retrieve", retrieve_node)
g.add_node("eval_each_doc", eval_each_doc_node)
g.add_node("rewrite_query", rewrite_query_node)
g.add_node("web_search", web_search_node)
g.add_node("refine", refine)
g.add_node("generate", generate)
g.add_node("ambigous", ambigous_node)

g.add_edge(START, "retrieve")
g.add_edge("retrieve", "eval_each_doc")

g.add_conditional_edges(
    "eval_each_doc",
    route_after_eval,
    {
        "refine": "refine",
        "web_search": "rewrite_query",
        "ambiguous": "ambiguous",
    },
)

g.add_edge("web_search", "eval_each_doc")
g.add_edge("refine", "generate")
g.add_edge("generate", END)
g.add_edge("ambiguous", END)

app = g.compile()
app

In [ ]:
res = app.invoke(
    {
        "question": "Recent AI news",
        "docs": [],
        "good_docs": [],
        "verdict": "",
        "reason": "",
        "strips": [],
        "kept_strips": [],
        "refined_context": "",
        "web_docs": [],
        "answer": "",
    }
)

print("VERDICT:", res["verdict"])
print("REASON:", res["reason"])
print("\nOUTPUT:\n", res["answer"])
print(res['web_query'])

### **Self-RAG**

In [ ]:
from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field
import time
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

docs = (
    PyPDFLoader("./documents/Company_Policies.pdf").load()
    + PyPDFLoader("./documents/Company_Profile.pdf").load()
    + PyPDFLoader("./documents/Product_and_Pricing.pdf").load()
)

chunks = RecursiveCharacterTextSplitter(
    chunk_size=600, chunk_overlap=150
).split_documents(docs)

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class State(TypedDict):
  question: str
  retrieval_query: str
  rewrite_tries: int
  need_retrieval: bool
  docs: List[Document]
  relevant_docs: List[Document]
  context: str
  answer: str
  issup: Literal["fully_supported", "partially_supported", "no_support"]
  evidence: List[str]
  retries: int
  isuse: Literal["useful", "not_useful"]
  use_reason: str
  web_query: str

class RetrieveDecision(BaseModel):
  should_retrieve: bool = Field(
    ...,
    description="True if external documents are needed to answer reliably, else False."
  )

decide_retrieval_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You decide whether retrieval is needed.\n"
      "Return JSON that matches this schema:\n"
      "{{'should_retrieve': boolean}}\n\n"
      "Guidelines:\n"
      "- should_retrieve=True if answering requires specific facts, citations, or info likely not in the model.\n"
      "- should_retrieve=False for general explanations, definitions, or reasoning that doesn't need sources.\n"
      "- If unsure, choose True."
    ),
    ("human", "Question: {question}"),
  ]
)

should_retrieve_llm = llm.with_structured_output(RetrieveDecision)

def decide_retrieval(state: State) -> State:
  decision: RetrieveDecision = should_retrieve_llm.invoke(
      decide_retrieval_prompt.format_prompt(question=state["question"])
  )
  return {"need_retrieval": decision.should_retrieve}

direct_generation_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "Answer the question using only your general knowledge.\n"
      "Do NOT assume access to external documents.\n"
      "If you are unsure or the answer requires specific sources, say:\n"
      "'I don't know based on my general knowledge.'"
    ),
    ("human", "{question}"),
  ]
)

def generate_direct(state: State) -> State:
  out = llm.invoke(direct_generation_prompt.format_prompt(question=state["question"]))
  return {"answer": out.content}

def retrieve(state: State):
  return {"docs": retriever.invoke(state["question"])}

class RelevanceDecision(BaseModel):
  is_relevant: bool = Field(
      ...,
      description="True if the document helps answer the question, else False."
  )

is_relevant_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are judging document relevance.\n"
      "Return JSON that matches this schema:\n"
      "{{'is_relevant': boolean}}\n\n"
      "A document is relevant if it contains information useful for answering the question."
    ),
    (
      "human",
      "Question:\n{question}\n\nDocument:\n{document}"
    ),
  ]
)

relevance_llm = llm.with_structured_output(RelevanceDecision)
def is_relevant(state: State):
  relevant_docs: List[Document] = []
  for doc in state["docs"]:
    decision: RelevanceDecision = relevance_llm.invoke(
        is_relevant_prompt.format_prompt(
          question=state["question"],
          document=doc.page_content,
      )
    )
    if decision.is_relevant:
      relevant_docs.append(doc)
  return {"relevant_docs": relevant_docs}

rag_generation_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are a business RAG assistant.\n"
      "Answer the user's question using ONLY the provided context.\n"
      "If the context does not contain enough information, say:\n"
      "'No relevant document found.'\n"
      "Do not use outside knowledge.\n"
    ),
    (
      "human",
      "Question:\n{question}\n\n"
      "Context:\n{context}\n"
    ),
  ]
)

def generate_from_context(state: State):
  context = "\n\n---\n\n".join(
    [d.page_content for d in state.get("relevant_docs", [])]
  ).strip()
  if not context:
    return {"answer": "No relevant document found.", "context": ""}
  out = llm.invoke(
    rag_generation_prompt.format_messages(
      question=state["question"],
      context=context
    )
  )
  return {"answer": out.content, "context": context}

class WebQuery(BaseModel):
  query: str

rewrite_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "Rewrite the user question into a web search query composed of keywords.\n"
      "Rules:\n"
      "- Keep it short (6–14 words).\n"
      "- If the question implies recency, add (last 30 days).\n"
      "- Do NOT answer the question.\n"
      "- Return JSON with a single key: query",
    ),
    ("human", "Question: {question}"),
  ]
)

rewrite_chain = rewrite_prompt | llm.with_structured_output(WebQuery)
def rewrite_query(state: State):
  out = rewrite_chain.invoke({"question": state["question"]})
  return {"web_query": out.query}

tavily = TavilySearchResults(max_results=5)
def web_search_node(state: State):
  q = state.get("web_query") or state["question"]
  results = tavily.invoke({"query": q})
  docs = []
  for r in results or []:
    title = r.get("title", "")
    url = r.get("url", "")
    content = r.get("content", "") or r.get("snippet", "")
    text = f"TITLE: {title}\nURL: {url}\nCONTENT:\n{content}"
    docs.append(Document(page_content=text, metadata={"source": "web", "url": url, "title": title}))
  return {"docs": docs}

def no_answer_found(state: State):
  return {"answer": "No relevant document found after all attempts.", "context": ""}

class IsSUPDecision(BaseModel):
  issup: Literal["fully_supported", "partially_supported", "no_support"]
  evidence: List[str] = Field(default_factory=list)

issup_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are verifying whether the ANSWER is supported by the CONTEXT.\n"
      "Return JSON with keys: issup, evidence.\n"
      "issup must be one of: fully_supported, partially_supported, no_support.\n\n"
      "How to decide issup:\n"
      "- fully_supported:\n"
      "  Every meaningful claim is explicitly supported by CONTEXT, and the ANSWER does NOT introduce\n"
      "  any qualitative/interpretive words that are not present in CONTEXT.\n"
      "  (Examples of disallowed words unless present in CONTEXT: culture, generous, robust, designed to,\n"
      "  supports professional development, best-in-class, employee-first, etc.)\n\n"
      "- partially_supported:\n"
      "  The core facts are supported, BUT the ANSWER includes ANY abstraction, interpretation, or qualitative\n"
      "  phrasing not explicitly stated in CONTEXT (e.g., calling policies 'culture', saying leave is 'generous',\n"
      "  or inferring outcomes like 'supports professional development').\n\n"
      "- no_support:\n"
      "  The key claims are not supported by CONTEXT.\n\n"
      "Rules:\n"
      "- Be strict: if you see ANY unsupported qualitative/interpretive phrasing, choose partially_supported.\n"
      "- If the answer is mostly unrelated to the question or unsupported, choose no_support.\n"
      "- Evidence: include up to 3 short direct quotes from CONTEXT that support the supported parts.\n"
      "- Do not use outside knowledge."
    ),
    (
      "human",
      "Question:\n{question}\n\n"
      "Answer:\n{answer}\n\n"
      "Context:\n{context}\n"
    ),
  ]
)

issup_llm = llm.with_structured_output(IsSUPDecision)
def is_sup(state: State):
  decision: IsSUPDecision = issup_llm.invoke(
    issup_prompt.format_prompt(
      question=state["question"],
      answer=state["answer"],
      context=state["context"],
    )
  )
  return {
    "issup": decision.issup,
    "evidence": decision.evidence,
  }

MAX_RETRIES = 10

def route_after_issup(state: State) -> Literal["accept_answer", "revise_answer"]:
  if state.get("issup") == "fully_supported":
    return "accept_answer"
  if state.get("retries", 0) >= MAX_RETRIES:
    return "accept_answer"
  return "revise_answer"

def accept_answer(state: State):
  return {}

revise_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are a STRICT reviser.\n\n"
      "You must output based on the following format:\n\n"
      "FORMAT (quote-only answer):
"
      "- <direct quote from the CONTEXT>\n"
      "- <direct quote from the CONTEXT>\n\n"
      "Rules:\n"
      "- Use ONLY the CONTEXT.\n"
      "- Do NOT add any new words besides bullet dashes and the quotes themselves.\n"
      "- Do NOT explain anything.\n"
      "- Do NOT say 'context', 'not mentioned', 'does not mention', 'not provided', etc.\n"
    ),
    (
      "human",
      "Question:\n{question}\n\n"
      "Current Answer:\n{answer}\n\n"
      "CONTEXT:\n{context}"
    ),
  ]
)

def revise_answer(state: State):
  out = llm.invoke(
      revise_prompt.format_prompt(
          question=state["question"],
          answer=state["answer"],
          context=state["context"],
      )
  )
  return {
    "answer": out.content,
    "retries": state.get("retries", 0) + 1,
  }

class IsUSEDecision(BaseModel):
  isuse: Literal["useful", "not_useful"]
  reason: str = Field(
      ...,
      description="Short reason in 1 line."
  )

isuse_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are judging USEFULNESS of the ANSWER for the QUESTION.\n\n"
      "Goal:\n"
      "- Decide if the answer actually addresses what the user asked.\n\n"
      "Return JSON with keys: isuse, reason.\n"
      "isuse must be one of: useful, not_useful.\n\n"
      "Rules:\n"
      "- useful: The answer directly answers the question or provides the requested specific info.\n"
      "- not_useful: The answer is generic, off-topic, or only gives related background without answering.\n"
      "- Do NOT use outside knowledge.\n"
      "- Do NOT re-check grounding (IsSUP already did that). Only check: 'Did we answer the question?'\n"
      "- Keep reason to 1 short line."
    ),
    (
      "human",
      "Question:\n{question}\n\nAnswer:\n{answer}"
    ),
  ]
)

isuse_llm = llm.with_structured_output(IsUSEDecision)

def is_use(state: State):
  decision: IsUSEDecision = isuse_llm.invoke(
    isuse_prompt.format_prompt(
      question=state["question"],
      answer=state.get("answer", ""),
    )
  )
  return {
    "isuse": decision.isuse,
    "use_reason": decision.reason,
  }

MAX_REWRITE_TRIES = 3

def route_after_isuse(state: State) -> Literal["END", "rewrite_question", "no_answer_found"]:
  if state.get("isuse") == "useful":
    return "END"
  if state.get("rewrite_tries", 0) >= MAX_REWRITE_TRIES:
    return "no_answer_found"
  return "rewrite_question"

class RewriteDecision(BaseModel):
  retrieval_query: str = Field(
      ...,
      description="Rewritten query optimized for vector retrieval against internal company PDFs."
  )

rewrite_for_retrieval_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "Rewrite the user's QUESTION into a query optimized for vector retrieval over INTERNAL company PDFs.\n\n"
      "Rules:\n"
      "- Keep it short (6–16 words).\n"
      "- Preserve key entities (e.g., NexaAI, plan names).\n"
      "- Add 2–5 high-signal keywords that likely appear in policy/pricing docs.\n"
      "- Remove filler words.\n"
      "- Do NOT answer the question.\n"
      "- Output JSON with key: retrieval_query\n\n"
      "Examples:\n"
      "Q: 'Do NexaAI plans include a free trial?'\n"
      "-> {{'retrieval_query': 'NexaAI free trial duration trial period plans'}}\n\n"
      "Q: 'What is NexaAI refund policy?'\n"
      "-> {{'retrieval_query': 'NexaAI refund policy cancellation refund timeline charges'}}"
    ),
    (
      "human",
      "QUESTION:\n{question}\n\n"
      "Previous retrieval query:\n{retrieval_query}\n\n"
      "Answer (if any):\n{answer}"
    ),
  ]
)

rewrite_llm = llm.with_structured_output(RewriteDecision)
def rewrite_question(state: State):
  decision: RewriteDecision = rewrite_llm.invoke(
    rewrite_for_retrieval_prompt.format_messages(
        question=state["question"],
        retrieval_query=state.get("retrieval_query", ""),
        answer=state.get("answer", "")
    )
  )
  return {
    "retrieval_query": decision.retrieval_query,
    "rewrite_tries": state.get("rewrite_tries", 0) + 1,
    "docs": [],
    "relevant_docs": [],
    "context": "",
  }

def route_after_decide(state: State) -> Literal["generate_direct", "retrieve"]:
  if state["need_retrieval"]:
    return "retrieve"
  return "generate_direct"

def route_after_relevance(state: State) -> Literal["generate_from_context", "rewrite_query"]:
  if state.get("relevant_docs") and len(state["relevant_docs"]) > 0:
    return "generate_from_context"
  return "rewrite_query"

g = StateGraph(State)
g.add_node("decide_retrieval", decide_retrieval)
g.add_node("generate_direct", generate_direct)
g.add_node("retrieve", retrieve)
g.add_node("is_relevant", is_relevant)
g.add_node("generate_from_context", generate_from_context)
g.add_node("no_answer_found", no_answer_found)
g.add_node("is_sup", is_sup)
g.add_node("accept_answer", accept_answer)
g.add_node("revise_answer", revise_answer)
g.add_node("is_use", is_use)
g.add_node("rewrite_question", rewrite_question)
g.add_node("rewrite_query", rewrite_query)
g.add_node("web_search", web_search_node)

g.add_edge(START, "decide_retrieval")
g.add_conditional_edges(
    "decide_retrieval",
    route_after_decide,
    {
        "generate_direct": "generate_direct",
        "retrieve": "retrieve",
    },
)
g.add_edge("generate_direct", END)
g.add_edge("retrieve", "is_relevant")

g.add_conditional_edges(
    "is_relevant",
    route_after_relevance,
    {
        "generate_from_context": "generate_from_context",
        "rewrite_query": "rewrite_query",
    },
)

g.add_edge("rewrite_query", "web_search")
g.add_edge("web_search", "is_relevant")
g.add_edge("generate_from_context", "is_sup")

g.add_conditional_edges(
    "is_sup",
    route_after_issup,
    {
        "accept_answer": "is_use",
        "revise_answer": "revise_answer",
    },
)
g.add_edge("revise_answer", "is_sup")

g.add_conditional_edges(
    "is_use",
    route_after_isuse,
    {
        "END": END,
        "rewrite_question": "rewrite_question",
        "no_answer_found": "no_answer_found",
    },
)
g.add_edge("rewrite_question", "retrieve")
g.add_edge("no_answer_found", END)

app = g.compile()
app

In [ ]:
initial_state = {
    "question": "Describe NexaAI’s company culture.",
    "retrieval_query": "What is the refund policy of NexaAI",
    "rewrite_tries": 0,
    "docs": [],
    "relevant_docs": [],
    "context": "",
    "answer": "",
    "issup": "",
    "evidence": [],
    "retries": 0,
    "isuse": "not_useful",
    "use_reason": "",
}


result = app.invoke(
    initial_state,
    config={"recursion_limit": 80},
)

print("\n===== RAG EXECUTION RESULT =====\n")
print("Question:", initial_state.get("question"))
print("Need Retrieval:", result.get("need_retrieval"))

print("Rewrite tries (retrieval):", result.get("rewrite_tries", 0))
print("Support revise tries:", result.get("retries", 0))

print("\nRetrieval:")
print("  Total retrieved docs:", len(result.get("docs", []) or []))
print("  Relevant docs:", len(result.get("relevant_docs", []) or []))

relevant_docs = result.get("relevant_docs", []) or []
if relevant_docs:
    print("\nRelevant docs (source/page):")
    for i, d in enumerate(relevant_docs, 1):
        src = (d.metadata or {}).get("source", "unknown")
        page = (d.metadata or {}).get("page", None)
        title = (d.metadata or {}).get("title", "")
        extra = f", title={title}" if title else ""
        if page is not None:
            print(f"  {i}. source={src}, page={page}{extra}")
        else:
            print(f"  {i}. source={src}{extra}")

print("\nVerification (IsSUP):")
print("  issup:", result.get("issup"))
evidence = result.get("evidence", []) or []
if evidence:
    print("  evidence:")
    for e in evidence:
        print("   -", e)
else:
    print("  evidence: (none)")

print("\nUsefulness (IsUSE):")
print("  isuse:", result.get("isuse"))
print("  reason:", result.get("use_reason", ""))

print("\nFinal Answer:")
print(result.get("answer"))

print("\n===============================\n")